# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout (`holdout.csv`), 5-fold stratified CV on training rows, selection by mean CV ROC AUC, holdout scored once.

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data.csv')
holdout_idx = pd.read_csv('holdout.csv')['row'].to_numpy()   # 0-based positions in data.csv
is_hold = np.zeros(len(df), bool); is_hold[holdout_idx] = True
train, hold = df[~is_hold], df[is_hold]
target = 'claim_within_year'
print(len(train), len(hold), train[target].mean(), hold[target].mean())

## Features
`device_serial` is excluded (identifier, per protocol). All other columns (channel, device age at sale, price, daily use at registration) are recorded at sale/registration, so known at prediction time.

In [ ]:
features = ['channel', 'device_age_years', 'price_eur', 'daily_hours']
cat, num = ['channel'], ['device_age_years', 'price_eur', 'daily_hours']
X, y = train[features], train[target]
print(train.groupby('channel')[target].mean())
print(train[num].corrwith(y))

## Candidates (all named before any is scored)
Preprocessing lives inside the pipelines, so it is fit within each CV fold.

In [ ]:
def prep(scale):
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat),
                              ('n', StandardScaler() if scale else 'passthrough', num)])
candidates = {
    'baseline_prior': make_pipeline(prep(False), DummyClassifier(strategy='prior')),
    'logreg_C1': make_pipeline(prep(True), LogisticRegression(C=1.0, max_iter=1000)),
    'logreg_C0.1': make_pipeline(prep(True), LogisticRegression(C=0.1, max_iter=1000)),
    'random_forest': make_pipeline(prep(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0)),
    'hist_gb': make_pipeline(prep(False), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
print(list(candidates))

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = {n: cross_val_score(m, X, y, cv=cv, scoring='roc_auc').tolist() for n, m in candidates.items()}
summary = pd.DataFrame({n: {'mean': np.mean(s), 'std': np.std(s)} for n, s in cv_scores.items()}).T
print(summary)
chosen = max(cv_scores, key=lambda n: np.mean(cv_scores[n]))
print('chosen:', chosen)

## Holdout (scored once)

In [ ]:
final = candidates[chosen].fit(X, y)
holdout_auc = roc_auc_score(hold[target], final.predict_proba(hold[features])[:, 1])
print(f'Holdout ROC AUC of {chosen}: {holdout_auc:.4f}')

In [ ]:
folds = [[[int(r) for r in train.index[tr]], [int(r) for r in train.index[va]]]
         for tr, va in cv.split(X, y)]
evidence = {
    'features': features,
    'candidates': list(candidates),
    'cv_folds': folds,
    'cv_scores': cv_scores,
    'chosen': chosen,
    'holdout_rows': [int(r) for r in hold.index],
    'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json', 'w'))

## Conclusion
See the CV table and holdout AUC above: an AUC near 0.5 would mean claims are not predictable from these features; higher values show how much signal sale-time attributes carry.